# Wykorzystanie Transfer Learning przy NLP + Fine-Tunng modelu
Zwykle, gdy staramy się wykorzystać transfer learning, mamy jakiś zestaw danych dotyczących naszego problemu i chcemy dopasować model do naszych danych, przy czym np. danych jest za mało, by wytrenować go od zera. Kolejną możliwością, oprócz wykorzystania czystego modelu z gotowymi wagami, jest znalezienie modelu i dopasowanie jego parametrów do naszego problemu. 

Standardowo zaczynamy od importu potrzebnych bibliotek, tym razem pracujemy z przetwarzaniem języka naturalnego i analizą nacechowania emocjonalnego tekstu. Konkretnie, chcemy ocenić czy opinia jest pozytywna czy negatywna. Będziemy korzystać z bazy danych IMDB oraz modelu BERT. 

In [21]:
import os
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
from transformers import BertTokenizer, BertModel, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import pandas as pd

W drugiej kolejności importujemy dane i ładujemy je do odpowiednich zmiennych

In [23]:
def load_imdb_data(data_file):
    df = pd.read_csv(data_file)
    texts = df['review'].tolist()
    labels = [1 if sentiment == "positive" else 0 for sentiment in df['sentiment'].tolist()]
    return texts, labels

data_file = "IMDBDataset.csv"
texts, labels = load_imdb_data(data_file)

Jako że chcemy działać z klasyfikacją tekstu, przygotowujemy swoją klasę do tego, w celu przygotowania danych do trenowania modelu. Poniższa metoda przeprowadzi tokenizację tekstu, jak równiez przygotuje całość pod potrzeby modelu. 

In [25]:
class TextClassificationDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):  
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]
        encoding = self.tokenizer(text, return_tensors='pt', max_length=self.max_length, padding='max_length', truncation=True)
        return {'input_ids': encoding['input_ids'].flatten(), 'attention_mask': encoding['attention_mask'].flatten(), 'label': torch.tensor(label)}

Druga klasa, którą przygotowujemy to nasza wersja modelu BERT. Strukturę tą budujemy na podstawie struktury oryginalnego modelu BERT, dodając warstwę Dropout i Linear do klasyfikacji tekstu. 

Z punktu widzenia modelu pobieramy ID i Maski przygotowane przez metody w poprzedniej klasie, następnie przepuszczamy je przez model BERT oraz nasze dodatkowe warsty, a następnie klasyfikator zwraca wynik swojego działania dla danego przykładu.

In [26]:
class BERTClassifier(nn.Module):
    def __init__(self, bert_model_name, num_classes):
        super(BERTClassifier, self).__init__()
        self.bert = BertModel.from_pretrained(bert_model_name)
        self.dropout = nn.Dropout(0.1)
        self.fc = nn.Linear(self.bert.config.hidden_size, num_classes)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = outputs.pooler_output
        x = self.dropout(pooled_output)
        logits = self.fc(x)
        return logits

Idąc dalej, potrzebujemy funkcji do trenowania modelu oraz metody jego ewaluacji - oba te elementy można zrealizować jako funkcje do wkorzystania w dalszej części programu. 

In [27]:
def train(model, data_loader, optimizer, scheduler, device):
    model.train()
    for batch in data_loader:
        optimizer.zero_grad()
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        loss = nn.CrossEntropyLoss()(outputs, labels)
        loss.backward()
        optimizer.step()
        scheduler.step()

def evaluate(model, data_loader, device):
    model.eval()
    predictions = []
    actual_labels = []
    with torch.no_grad():
        for batch in data_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            _, preds = torch.max(outputs, dim=1)
            predictions.extend(preds.cpu().tolist())
            actual_labels.extend(labels.cpu().tolist())
    return accuracy_score(actual_labels, predictions), classification_report(actual_labels, predictions)

Kolejnym elementem jest metoda, która będzie nam służyła do predykccji nacechowania emocjonalnego opinii z bazy danych (sentiment analysis). Funkcja liczy również dokładność i wypisuje raport klasyfikacji, w celu pokazania jakości działania modelu. 

In [28]:
def predict_sentiment(text, model, tokenizer, device, max_length=128):
    model.eval()
    encoding = tokenizer(text, return_tensors='pt', max_length=max_length, padding='max_length', truncation=True)
    input_ids = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)

    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        _, preds = torch.max(outputs, dim=1)
    return "positive" if preds.item() == 1 else "negative"

Teraz możemy przejść do faktycznego przygotowania i wytrenowania modelu, zaczynając od zdefiniowania jego głównych parametrów. 

In [29]:
# Set up parameters
bert_model_name = 'bert-base-uncased'
num_classes = 2
max_length = 128
batch_size = 16
num_epochs = 4
learning_rate = 2e-5

W kolejnych krokach wczytujemy dane, inicializujemy tokenizera, zbiór danych data loader, ustawiamy urządzenie na którym będziemy przeprowadzać obliczenia oraz sam model, plus optymizator i learning rate dla modelu.

In [30]:
train_texts, val_texts, train_labels, val_labels = train_test_split(texts, labels, test_size=0.2, random_state=42)

tokenizer = BertTokenizer.from_pretrained(bert_model_name)
train_dataset = TextClassificationDataset(train_texts, train_labels, tokenizer, max_length)
val_dataset = TextClassificationDataset(val_texts, val_labels, tokenizer, max_length)
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=batch_size)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = BERTClassifier(bert_model_name, num_classes).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
total_steps = len(train_dataloader) * num_epochs
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=total_steps)

Następnym krokiem jest wytrenowanie modelu na przygotowanych danych.

### Uwaga!
To trochę trwa, ilość epok jest mała nie bez powodu (u mnie, przy pełnym wykorzystaniu GPU - NVIDIA GeForce RTX 5070 Ti, wersja laptopowa - trenowanie zajęło ponad 20 minut). Dlatego też model jest zapisywany od razu po wytrenowaniu (w razie czego zamiast trenować, można pominąć poniższą funkcję i zamiast tego odkomentować fragment kodu w następnym przykładzie, ładując wytrenowany model i wykorzystać go do predykcji)

In [31]:
for epoch in range(num_epochs):
    print(f"Epoch {epoch + 1}/{num_epochs}")
    train(model, train_dataloader, optimizer, scheduler, device)
    accuracy, report = evaluate(model, val_dataloader, device)
    print(f"Validation Accuracy: {accuracy:.4f}")
    print(report)

torch.save(model.state_dict(), "bert_classifier.pth")
torch.save(model, "bert_full")

Epoch 1/4
Validation Accuracy: 0.8878
              precision    recall  f1-score   support

           0       0.87      0.91      0.89      4961
           1       0.91      0.87      0.89      5039

    accuracy                           0.89     10000
   macro avg       0.89      0.89      0.89     10000
weighted avg       0.89      0.89      0.89     10000

Epoch 2/4
Validation Accuracy: 0.8928
              precision    recall  f1-score   support

           0       0.91      0.87      0.89      4961
           1       0.88      0.92      0.90      5039

    accuracy                           0.89     10000
   macro avg       0.89      0.89      0.89     10000
weighted avg       0.89      0.89      0.89     10000

Epoch 3/4
Validation Accuracy: 0.8959
              precision    recall  f1-score   support

           0       0.92      0.86      0.89      4961
           1       0.87      0.93      0.90      5039

    accuracy                           0.90     10000
   macro avg  

Po wytrenowaniu modelu możemy przejść do predykcji na wybranych przykładach. 

In [33]:
#Optional model load
#model = torch.load("bert_full", weights_only=False)

# Test sentiment prediction
test_text = "The movie was great and I really enjoyed the performances of the actors."
sentiment = predict_sentiment(test_text, model, tokenizer, device)
print("The movie was great and I really enjoyed the performances of the actors.")
print(f"Predicted sentiment: {sentiment}")

 # Test sentiment prediction
test_text = "Worst movie of the year."
sentiment = predict_sentiment(test_text, model, tokenizer, device)
print("Worst movie of the year.")
print(f"Predicted sentiment: {sentiment}")

The movie was great and I really enjoyed the performances of the actors.
Predicted sentiment: positive
Worst movie of the year.
Predicted sentiment: negative


Można oczywiście pobawić się dalej z testowaniem modelu, jednak na powyższych przykładach, oraz po parametrych wypisywanych przy każdej epoce trenowania modelu widać, że całkiem nieźle radzi sobie z zadaniem. Z punktu wdzenia poziomu skomplikowania problemu, dokładność w granicach 89% wygląda bardzo dobrze, nie mówiąc o tym, że pozostałe parametry również potwierdzają, że model dobrze radzi sobie z problemem.

Jak łatwo zauważyć przy większych modelach nawet proces dotrenowywania może być czasochłonny - nasz model zapewne osiągnąłby większą dokładność, gdyby zamiast 4-rech epok zapuścić trening np. na 50 lub 100, przy czym przy tego typu, dłuższym trenowaniu warto również ustawić kryterium early stop, by zatrzymać trening i zapisać najlepiej działający model z ostatnich iteracji, w momencie gdy poprawa jakości modelu spada poniżej wybranego poziomu - podobnie jak przy innych rozwiązaniach, zwiększa się wtedy ryzyko przetrenowania modelu. 

Oczwistym problemem jest, że nawet przy gotowych modelach które jedynie dotrenowujemy do naszego zastosowania, trening może trochę zająć. Przy dużych modelach nie jest niczym niezwykłym, że dotrenowanie modelu zajmuje np. kilka godzin czy dni. Ze względu na swoje działanie i możliwość wychwycenia ciężkich do znalezienia wzorców w danych, sieci neuronowe są jednak szeroko stosowane i ich zastosowania jedynie się rozszerzają. 

Warto oczywiście zauważyć, że gdybyśmy próbowali wytrenować podobny model od zera, trening również byłby bardzo dlugi, a zapewne nie osiągnęlibyśmy równie wysokiej dokładności predykcji. To ogólna uwaga odnośnie przygotowywania rozwiązań AI: jeżeli istnieje pretrenowany model do tego co próbujemy zrobić, albo czegoś podobnego (np. operującego na podobnych danych), w pierwszej kolejności warto sprawdzić te modele i w razie potrzeby dopasować i dotrenować do naszego problemu. Tworzenie całego modelu od zera jest raczej ostatecznością. 

Niezależnie, zawsze warto zacząć od przebadania tematu i wyszukania aktualnych trendów/stosowanych rozwiązań, jako że może nam to oszczędzić sporo pracy i problemów. 